<a href="https://colab.research.google.com/github/deji4things2000/mlpro/blob/master/CNN_Representational_Power.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Model Implementation

In [ ]:
import random
import numpy as np
import torch

SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class RandomCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.fc = nn.Linear(32 * 7 * 7, 128)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = x.view(x.size(0), -1)
        return self.fc(x)


Freeze Weights

In [3]:
model = RandomCNN()
for p in model.parameters():
    p.requires_grad = False
model.eval()


RandomCNN(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc): Linear(in_features=1568, out_features=128, bias=True)
)

Feature Extraction

In [4]:
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
import numpy as np

transform = transforms.ToTensor()

train_ds = torchvision.datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
test_ds = torchvision.datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

def extract_features(loader):
    features, labels = [], []
    with torch.no_grad():
        for x, y in loader:
            z = model(x)
            features.append(z.numpy())
            labels.append(y.numpy())
    return np.vstack(features), np.hstack(labels)

X_train, y_train = extract_features(train_loader)
X_test, y_test = extract_features(test_loader)


100%|██████████| 9.91M/9.91M [00:00<00:00, 61.4MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.64MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 15.0MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.17MB/s]


Train Linear SVM

In [5]:
from sklearn.svm import LinearSVC

svm = LinearSVC(C=1.0, max_iter=5000)
svm.fit(X_train, y_train)

acc = svm.score(X_test, y_test)
print(f"Test accuracy: {acc:.4f}")


Test accuracy: 0.9425
